<a href="https://colab.research.google.com/github/SantiagoAmaya99/Crack-Detector-Using-Deep-Learning-Models/blob/main/Prueba_de_modelos_Detecci%C3%B3n_de_grietas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd '/content/drive/MyDrive/Dataset_IA2'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Dataset_IA2


In [16]:
#@title Librerias
import tensorflow as tf
import pandas as pd
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedGroupKFold
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications import ConvNeXtTiny
import os, re, shutil
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
import numpy as np

In [3]:
import os
import glob

# Definición de variables del dataset
base_dir = '.'
materiales = ['Pavements', 'Walls', 'Decks']
clases = ['Cracked', 'Non-cracked']

rows = []
for material in materiales:
    for clase in clases:
        folder_path = os.path.join(base_dir, material, clase)
        for img_path in glob.glob(os.path.join(folder_path, '*.*')):
            fname = os.path.basename(img_path)
            rows.append({
                "path": img_path,
                "label": 1 if clase == 'Cracked' else 0,
                "surface": material,
                "group": f"{material}_{fname.split("-")[0]}",
            })
df = pd.DataFrame(rows)

def group_split(frame, n_splits, seed=42):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    a, b = next(sgkf.split(frame, frame["label"], frame["group"]))
    return frame.iloc[a], frame.iloc[b]

trainval_df, test_df = group_split(df, 7)       # ~14% test
train_df, val_df = group_split(trainval_df, 6)  # ~14% val

# Verificación anti-fuga
assert set(train_df.group).isdisjoint(val_df.group)
assert set(train_df.group).isdisjoint(test_df.group)
assert set(val_df.group).isdisjoint(test_df.group)

for n, p in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(n, len(p), f"{p["label"].mean():.3f} con grieta")

train 40179 0.144 con grieta
val 8118 0.164 con grieta
test 7795 0.177 con grieta


In [4]:
for n, p in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(n, p['group'].nunique(), "fotos |", len(p), "imgs |",
          f"{p['label'].mean():.3f} con grieta")
    print(p['surface'].value_counts().to_dict())

train 165 fotos | 40179 imgs | 0.144 con grieta
{'Pavements': 18250, 'Walls': 11837, 'Decks': 10092}
val 33 fotos | 8118 imgs | 0.164 con grieta
{'Walls': 3276, 'Pavements': 2574, 'Decks': 2268}
test 32 fotos | 7795 imgs | 0.177 con grieta
{'Pavements': 3510, 'Walls': 3025, 'Decks': 1260}


#Modelo 1: CNN Propia


In [ ]:
def build_custom_cnn(input_shape=(128, 128, 3)):
    # CNN propia sin preentrenar
    model = models.Sequential([
        # Bloque 1
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.BatchNormalization(), # Regularización BatchNorm
        layers.MaxPooling2D((2, 2)),

        # Bloque 2
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Bloque 3
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        # Global Average Pooling (GAP) reemplaza al Flatten tradicional
        layers.GlobalAveragePooling2D(),

        # Regularización solicitada
        layers.Dropout(0.4),

        # Capa de salida binaria (1 neurona, sigmoide para clasificar grieta/no grieta)
        layers.Dense(1, activation='sigmoid')
    ])
    return model

model = build_custom_cnn()

#ResNET 50

In [5]:
#@title Modelo base
def build_resnet50_base(input_shape=(128, 128, 3)):
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # 2. Construir el nuevo modelo agregando la cabeza personalizada
    modelResNET = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4), # Requisito de regularización base para los experimentos
        layers.Dense(1, activation='sigmoid') # Salida binaria (grieta / no grieta)
    ])

    return modelResNET, base_model

# Crear la arquitectura inicial
modelResNET, base_model = build_resnet50_base()
modelResNET.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         2,049 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,589,761 (89.99 MB)

 Trainable params: 23,536,641 (89.79 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [11]:
# Rutas
RUTA_DRIVE = Path('/content/dataset_crudo')
RUTA_LOCAL = Path('/content/dataset_local')

def id_foto(nombre_archivo):
    m = re.search(r'(\d+)[-_]\d+', Path(nombre_archivo).stem)
    return m.group(1) if m else Path(nombre_archivo).stem

registros = []

for material in ['Decks', 'Pavements', 'Walls']:
    for clase, etiqueta in [('Cracked', 1), ('Non-cracked', 0)]:
        carpeta_origen = RUTA_DRIVE / material / clase
        carpeta_dest = RUTA_LOCAL / clase
        carpeta_dest.mkdir(parents=True, exist_ok=True)

        if carpeta_origen.exists():
            # CAMBIO CLAVE: iterdir() atrapa todo, ignorando problemas de extensión
            archivos = [f for f in carpeta_origen.iterdir() if f.is_file()]

            for f in tqdm(archivos, desc=f'{material}/{clase}'):
                dest = carpeta_dest / f'{material}_{f.name}'
                if not dest.exists():
                    shutil.copy(f, dest)

                registros.append({
                    'ruta': str(dest),
                    'etiqueta': etiqueta,
                    'grupo': f'{material}_{id_foto(f.name)}'
                })

df = pd.DataFrame(registros)
print(f"Total de imágenes cargadas correctamente: {len(df)}")

Decks/Cracked:   0%|          | 0/2025 [00:00<?, ?it/s]

Decks/Non-cracked:   0%|          | 0/11595 [00:00<?, ?it/s]

Pavements/Cracked:   0%|          | 0/2608 [00:00<?, ?it/s]

Pavements/Non-cracked:   0%|          | 0/21726 [00:00<?, ?it/s]

Walls/Cracked:   0%|          | 0/3851 [00:00<?, ?it/s]

Walls/Non-cracked:   0%|          | 0/14287 [00:00<?, ?it/s]

Total de imágenes cargadas correctamente: 56092


In [10]:
!cp -r "/content/drive/MyDrive/Dataset_IA2" "/content/dataset_crudo"

In [13]:
from sklearn.model_selection import GroupShuffleSplit

SEMILLA = 42 # Faltaba definir esta variable

def dividir_por_grupo(datos, fraccion, semilla):
    gss = GroupShuffleSplit(n_splits=1, test_size=fraccion, random_state=semilla)
    idx_a, idx_b = next(gss.split(datos, groups=datos['grupo']))
    return datos.iloc[idx_a].reset_index(drop=True), datos.iloc[idx_b].reset_index(drop=True)

df_trainval, df_test = dividir_por_grupo(df, 0.15, SEMILLA)
df_train, df_val = dividir_por_grupo(df_trainval, 0.15 / 0.85, SEMILLA)

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

Train: 39046 | Val: 8478 | Test: 8568


In [14]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

TAM_IMG = (128, 128)
TAM_LOTE = 64
AUTOTUNE = tf.data.AUTOTUNE

n_total = len(df_train)
n1 = int(df_train.etiqueta.sum())
n0 = n_total - n1
PESO_CLASE = {0: n_total / (2 * n0), 1: n_total / (2 * n1)}

augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.1, fill_mode='reflect'),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
])

def cargar_imagen(ruta, etiqueta):
    img = tf.io.read_file(ruta)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, TAM_IMG)
    return img, tf.cast(etiqueta, tf.float32)

def crear_dataset(datos, entrenamiento=False):
    rutas = datos['ruta'].astype(str).values
    etiquetas = datos['etiqueta'].values.astype('int32')
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))

    if entrenamiento:
        ds = ds.shuffle(len(datos), seed=SEMILLA, reshuffle_each_iteration=True)

    ds = ds.map(cargar_imagen, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (x / 255.0, y), num_parallel_calls=AUTOTUNE)

    if entrenamiento:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
        w0, w1 = PESO_CLASE[0], PESO_CLASE[1]
        ds = ds.map(lambda x, y: (x, y, tf.where(y > 0.5, w1, w0)), num_parallel_calls=AUTOTUNE)

    return ds.batch(TAM_LOTE).prefetch(AUTOTUNE)

ds_train = crear_dataset(df_train, entrenamiento=True)
ds_val = crear_dataset(df_val)
ds_test = crear_dataset(df_test)

In [21]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import os

# 1. Limpieza de seguridad: Filtrar imágenes vacías o corruptas de 0 bytes
def filtrar_imagenes_validas(datos):
    rutas_validas = []
    for r in datos['ruta'].values:
        if os.path.exists(r) and os.path.getsize(r) > 0:
            rutas_validas.append(True)
        else:
            rutas_validas.append(False)
    return datos[rutas_validas].reset_index(drop=True)

# Filtrar DataFrames
df_train_limpio = filtrar_imagenes_validas(df_train)
df_val_limpio = filtrar_imagenes_validas(df_val)

# Regenerar datasets con datos limpios
ds_train = crear_dataset(df_train_limpio, entrenamiento=True)
ds_val = crear_dataset(df_val_limpio)

# 2. Congelar toda la ResNet50
base_model.trainable = False

# 3. Compilar (LR = 1e-3)
modelResNET.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 4. Configurar los callbacks optimizados para validación
callbacks_resnet_exp1 = [
    EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        'resnet50_feature_extraction.h5',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

# 5. Ejecutar el entrenamiento de la ResNet50 (Feature Extraction)
history_resnet_exp1 = modelResNET.fit(
    ds_train,
    validation_data=ds_val,
    epochs=30,
    callbacks=callbacks_resnet_exp1
)

Epoch 1/30
611/611 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step - accuracy: 0.4915 - auc: 0.4985 - loss: 0.6961
Epoch 1: val_auc improved from None to 0.60284, saving model to resnet50_feature_extraction.h5



Epoch 1: finished saving model to resnet50_feature_extraction.h5
611/611 ━━━━━━━━━━━━━━━━━━━━ 227s 351ms/step - accuracy: 0.4999 - auc: 0.5055 - loss: 0.6948 - val_accuracy: 0.8401 - val_auc: 0.6028 - val_loss: 0.6745 - learning_rate: 0.0010
Epoch 2/30
610/611 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - accuracy: 0.5096 - auc: 0.5128 - loss: 0.6933
Epoch 2: val_auc improved from 0.60284 to 0.64835, saving model to resnet50_feature_extraction.h5



Epoch 2: finished saving model to resnet50_feature_extraction.h5
611/611 ━━━━━━━━━━━━━━━━━━━━ 201s 329ms/step - accuracy: 0.5206 - auc: 0.5137 - loss: 0.6943 - val_accuracy: 0.1607 - val_auc: 0.6483 - val_loss: 0.7027 - learning_rate: 0.0010
Epoch 3/30
610/611 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step - accuracy: 0.5000 - auc: 0.5113 - loss: 0.6990
Epoch 3: val_auc improved from 0.64835 to 0.65748, saving model to resnet50_feature_extraction.h5



Epoch 3: finished saving model to resnet50_feature_extraction.h5
611/611 ━━━━━━━━━━━━━━━━━━━━ 201s 328ms/step - accuracy: 0.5368 - auc: 0.5147 - loss: 0.6936 - val_accuracy: 0.7374 - val_auc: 0.6575 - val_loss: 0.6898 - learning_rate: 0.0010
Epoch 4/30
610/611 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step - accuracy: 0.4586 - auc: 0.5212 - loss: 0.6961
Epoch 4: val_auc improved from 0.65748 to 0.66248, saving model to resnet50_feature_extraction.h5



Epoch 4: finished saving model to resnet50_feature_extraction.h5
611/611 ━━━━━━━━━━━━━━━━━━━━ 204s 333ms/step - accuracy: 0.5066 - auc: 0.5185 - loss: 0.6933 - val_accuracy: 0.8287 - val_auc: 0.6625 - val_loss: 0.6691 - learning_rate: 0.0010
Epoch 5/30
610/611 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step - accuracy: 0.5518 - auc: 0.5124 - loss: 0.6932
Epoch 5: val_auc did not improve from 0.66248
611/611 ━━━━━━━━━━━━━━━━━━━━ 196s 321ms/step - accuracy: 0.5234 - auc: 0.5133 - loss: 0.6935 - val_accuracy: 0.8310 - val_auc: 0.6545 - val_loss: 0.6622 - learning_rate: 0.0010
Epoch 6/30
610/611 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - accuracy: 0.5076 - auc: 0.5181 - loss: 0.6938
Epoch 6: val_auc did not improve from 0.66248
611/611 ━━━━━━━━━━━━━━━━━━━━ 200s 327ms/step - accuracy: 0.5262 - auc: 0.5201 - loss: 0.6926 - val_accuracy: 0.8273 - val_auc: 0.6571 - val_loss: 0.6603 - learning_rate: 0.0010
Epoch 7/30
610/611 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step - accuracy: 0.5226 - auc: 0.5279 - loss: 0.6916
Epoch


Epoch 9: finished saving model to resnet50_feature_extraction.h5

Epoch 9: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
611/611 ━━━━━━━━━━━━━━━━━━━━ 205s 336ms/step - accuracy: 0.5359 - auc: 0.5249 - loss: 0.6921 - val_accuracy: 0.2911 - val_auc: 0.6682 - val_loss: 0.7025 - learning_rate: 0.0010
Epoch 10/30
610/611 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step - accuracy: 0.5058 - auc: 0.5218 - loss: 0.6883
Epoch 10: val_auc did not improve from 0.66822
611/611 ━━━━━━━━━━━━━━━━━━━━ 204s 333ms/step - accuracy: 0.5123 - auc: 0.5293 - loss: 0.6911 - val_accuracy: 0.6932 - val_auc: 0.6660 - val_loss: 0.6893 - learning_rate: 5.0000e-04
Epoch 11/30
610/611 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - accuracy: 0.5346 - auc: 0.5291 - loss: 0.6869
Epoch 11: val_auc improved from 0.66822 to 0.66870, saving model to resnet50_feature_extraction.h5



Epoch 11: finished saving model to resnet50_feature_extraction.h5
611/611 ━━━━━━━━━━━━━━━━━━━━ 202s 331ms/step - accuracy: 0.5285 - auc: 0.5272 - loss: 0.6907 - val_accuracy: 0.1701 - val_auc: 0.6687 - val_loss: 0.7097 - learning_rate: 5.0000e-04
Epoch 12/30
610/611 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step - accuracy: 0.4836 - auc: 0.5360 - loss: 0.6902
Epoch 12: val_auc did not improve from 0.66870

Epoch 12: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
611/611 ━━━━━━━━━━━━━━━━━━━━ 201s 329ms/step - accuracy: 0.5203 - auc: 0.5325 - loss: 0.6904 - val_accuracy: 0.8057 - val_auc: 0.6656 - val_loss: 0.6708 - learning_rate: 5.0000e-04
Epoch 13/30
610/611 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step - accuracy: 0.5647 - auc: 0.5321 - loss: 0.6842
Epoch 13: val_auc did not improve from 0.66870
611/611 ━━━━━━━━━━━━━━━━━━━━ 200s 327ms/step - accuracy: 0.5298 - auc: 0.5334 - loss: 0.6900 - val_accuracy: 0.7991 - val_auc: 0.6672 - val_loss: 0.6747 - learning_rate: 2.5000e-04
Epoch 14/30



Epoch 15: finished saving model to resnet50_feature_extraction.h5

Epoch 15: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
611/611 ━━━━━━━━━━━━━━━━━━━━ 207s 338ms/step - accuracy: 0.5574 - auc: 0.5411 - loss: 0.6895 - val_accuracy: 0.1612 - val_auc: 0.6708 - val_loss: 0.7167 - learning_rate: 2.5000e-04
Epoch 16/30
610/611 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - accuracy: 0.4384 - auc: 0.5455 - loss: 0.6874
Epoch 16: val_auc did not improve from 0.67083
611/611 ━━━━━━━━━━━━━━━━━━━━ 198s 324ms/step - accuracy: 0.4881 - auc: 0.5398 - loss: 0.6894 - val_accuracy: 0.7503 - val_auc: 0.6677 - val_loss: 0.6863 - learning_rate: 1.2500e-04
Epoch 17/30
610/611 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step - accuracy: 0.5142 - auc: 0.5454 - loss: 0.6914
Epoch 17: val_auc did not improve from 0.67083
611/611 ━━━━━━━━━━━━━━━━━━━━ 197s 322ms/step - accuracy: 0.5514 - auc: 0.5430 - loss: 0.6893 - val_accuracy: 0.7961 - val_auc: 0.6669 - val_loss: 0.6767 - learning_rate: 1.2500e-04
Epoch 18/30



Epoch 20: finished saving model to resnet50_feature_extraction.h5
611/611 ━━━━━━━━━━━━━━━━━━━━ 197s 323ms/step - accuracy: 0.5787 - auc: 0.5409 - loss: 0.6893 - val_accuracy: 0.7982 - val_auc: 0.6709 - val_loss: 0.6752 - learning_rate: 6.2500e-05
Epoch 21/30
610/611 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step - accuracy: 0.5619 - auc: 0.5438 - loss: 0.6879
Epoch 21: val_auc did not improve from 0.67087

Epoch 21: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.
611/611 ━━━━━━━━━━━━━━━━━━━━ 203s 324ms/step - accuracy: 0.5457 - auc: 0.5415 - loss: 0.6893 - val_accuracy: 0.8099 - val_auc: 0.6679 - val_loss: 0.6669 - learning_rate: 6.2500e-05
Epoch 22/30
610/611 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step - accuracy: 0.6007 - auc: 0.5447 - loss: 0.6903
Epoch 22: val_auc did not improve from 0.67087
611/611 ━━━━━━━━━━━━━━━━━━━━ 199s 326ms/step - accuracy: 0.5873 - auc: 0.5420 - loss: 0.6892 - val_accuracy: 0.8047 - val_auc: 0.6675 - val_loss: 0.6704 - learning_rate: 3.1250e-05
Epoch 23/30


In [ ]:
#@title Ultima capa descongelada


# 1. Permitir entrenamiento en la base
base_model.trainable = True

# 2. Iterar y congelar todo excepto el último bloque (sin tocar los BN)
for layer in base_model.layers:
    # Si la capa pertenece al bloque 5 y NO es BatchNormalization, se descongela
    if layer.name.startswith('conv5_block') and not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True
    else:
        # Congelamos el resto de bloques y TODAS las capas BatchNormalization
        layer.trainable = False

# 3. Compilar (LR = 1e-4)
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

In [ ]:
#@title Modelo descongelado a excepción de BN


# 1. Permitir entrenamiento en la base
base_model.trainable = True

# 2. Congelar EXCLUSIVAMENTE las capas BatchNormalization
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

# 3. Compilar (LR = 1e-5 y adición de weight decay)
# Nota: En Keras modernos, el optimizador Adam acepta el parámetro weight_decay directamente
model.compile(
    optimizer=Adam(learning_rate=1e-5, weight_decay=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

#EfficientNetB0

In [ ]:
def build_efficientnet_base(input_shape=(128, 128, 3)):
    # 1. Instanciar EfficientNetB0 preentrenada en ImageNet
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # 2. Construir la cabeza personalizada
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4), # Regularización requerida
        layers.Dense(1, activation='sigmoid') # Clasificación binaria
    ])

    return model, base_model

model, base_model = build_efficientnet_base()

In [ ]:
#@title Modelo completamente congelado


# 1. Congelar toda la base
base_model.trainable = False

# 2. Compilar con LR = 1e-3
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Ejecutas model.fit(..., class_weight=class_weight_dict)

In [ ]:
#@title Descongelamiento gradual

# 1. Permitir entrenamiento en la base
base_model.trainable = True

# 2. Descongelar solo el último bloque (block7) y dejar todo lo demás congelado
for layer in base_model.layers:
    if layer.name.startswith('block7') and not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True
    else:
        layer.trainable = False

# 3. Compilar con LR = 1e-4
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
# Ejecutas model.fit(...) unas cuantas épocas

In [ ]:
#@title Completamente descongelado
# 1. Permitir entrenamiento en la base
base_model.trainable = True

# 2. Congelar EXCLUSIVAMENTE las capas BatchNormalization
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

# 3. Compilar con LR = 1e-5 y agregar weight_decay
model.compile(
    optimizer=Adam(learning_rate=1e-5, weight_decay=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
# Ejecutas model.fit(...)

#VGG16

In [ ]:
base_vgg = VGG16(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
model_vgg = models.Sequential([
    base_vgg,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5), # Especificado en la tabla
    layers.Dense(1, activation='sigmoid')
])

# 2. Descongelar solo los últimos 2 bloques (block4 y block5)
base_vgg.trainable = True
for layer in base_vgg.layers:
    if layer.name.startswith('block4') or layer.name.startswith('block5'):
        layer.trainable = True
    else:
        layer.trainable = False

# 3. Compilación (LR = 1e-4)
model_vgg.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

#DenseNet121

In [ ]:
# 1. Base y arquitectura
base_dense = DenseNet121(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
model_dense = models.Sequential([
    base_dense,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4), # Especificado en la tabla
    layers.Dense(1, activation='sigmoid')
])

# 2. Descongelar último bloque (conv5) protegiendo el BN
base_dense.trainable = True
for layer in base_dense.layers:
    if layer.name.startswith('conv5') and not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True
    else:
        layer.trainable = False

# 3. Compilación (LR = 1e-4)
model_dense.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

#ConvNeXt

In [ ]:
# 1. Arquitectura con Data Augmentation integrado
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2)
])

base_convnext = ConvNeXtTiny(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
model_convnext = models.Sequential([
    data_augmentation,
    base_convnext,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4), # Dropout general
    layers.Dense(1, activation='sigmoid')
])

# 2. Fine-tuning completo protegiendo LayerNormalization
base_convnext.trainable = True
for layer in base_convnext.layers:
    if isinstance(layer, tf.keras.layers.LayerNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

# 3. Compilación (LR = 1e-5 y Weight Decay)
model_convnext.compile(
    optimizer=Adam(learning_rate=1e-5, weight_decay=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

MobileNetV3

In [ ]:
def build_mobilenetv3_base(input_shape=(128, 128, 3)):
    # 1. Instanciar MobileNetV3 (Large) preentrenada en ImageNet
    base_model = MobileNetV3Large(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    # 2. Construir la cabeza personalizada
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3), # Regularización requerida en la tabla
        layers.Dense(1, activation='sigmoid') # Clasificación binaria
    ])

    return model, base_model

model, base_model = build_mobilenetv3_base()

In [ ]:
# 1. Congelar toda la base
base_model.trainable = False

# 2. Compilar (LR = 1e-3)
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

In [ ]:
# 1. Permitir entrenamiento en la base
base_model.trainable = True

# 2. Descongelar últimos bloques (ej. expanded_conv_12 al 14 y Conv_1/Conv_2 finales)
for layer in base_model.layers:
    # Identificar las capas finales de MobileNetV3Large
    if ('expanded_conv_12' in layer.name or
        'expanded_conv_13' in layer.name or
        'expanded_conv_14' in layer.name or
        'Conv_1' in layer.name or
        'Conv_2' in layer.name):

        # Congelar BatchNormalization dentro de estos bloques
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True
    else:
        layer.trainable = False # Congelar el resto de la red temprana

# 3. Compilar (LR = 1e-4)[cite: 16]
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

In [ ]:
# 1. Permitir entrenamiento en la base
base_model.trainable = True

# 2. Congelar EXCLUSIVAMENTE las capas BatchNormalization
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

# 3. Compilar con LR = 1e-5 y agregar weight_decay
model.compile(
    optimizer=Adam(learning_rate=1e-5, weight_decay=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)